# Week 3 Day 1 — The Svensson Function

Tests for `svensson_zero_rate` in `src/termstructure/curves/svensson.py`.

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
from termstructure.curves.svensson import svensson_zero_rate

## 1. Asymptote checks

- `tau → 0`  should give `beta0 + beta1`
- `tau → ∞`  should give `beta0`

In [ ]:
b0, b1, b2, b3, l1, l2 = 0.04, -0.01, 0.02, -0.01, 2.0, 5.0

short = svensson_zero_rate(1e-10, b0, b1, b2, b3, l1, l2)
long  = svensson_zero_rate(1e6,   b0, b1, b2, b3, l1, l2)

print(f"tau->0:   {short:.8f}  expected {b0+b1:.8f}  match={abs(short-(b0+b1))<1e-6}")
print(f"tau->inf: {long:.8f}  expected {b0:.8f}  match={abs(long-b0)<1e-4}")

## 2. Scalar vs. array input

In [ ]:
scalar_result = svensson_zero_rate(10.0, b0, b1, b2, b3, l1, l2)
array_result  = svensson_zero_rate(np.array([10.0]), b0, b1, b2, b3, l1, l2)

print(f"Scalar input → type: {type(scalar_result).__name__}, value: {scalar_result:.6f}")
print(f"Array input  → type: {type(array_result).__name__}, value: {array_result[0]:.6f}")
assert isinstance(scalar_result, float), "scalar input should return float"
assert isinstance(array_result, np.ndarray), "array input should return ndarray"
print("Types OK")

## 3. Beta0 is the only free parameter at long maturities

Change beta1/beta2/beta3 while keeping beta0 fixed — the 30Y rate should barely move.

In [ ]:
tau_long = 30.0
base = svensson_zero_rate(tau_long, 0.04, -0.01,  0.02, -0.01, 2.0, 5.0)
var1 = svensson_zero_rate(tau_long, 0.04,  0.05,  0.10,  0.10, 2.0, 5.0)  # big beta1/2/3
var2 = svensson_zero_rate(tau_long, 0.04, -0.05, -0.10, -0.10, 2.0, 5.0)  # opposite sign

print(f"base   30Y: {base*100:.4f}%")
print(f"var1   30Y: {var1*100:.4f}%  (big +beta1/2/3)")
print(f"var2   30Y: {var2*100:.4f}%  (big -beta1/2/3)")
print(f"All near 4%: {all(abs(r - 0.04) < 0.001 for r in [base, var1, var2])}")

## 4. Lambda controls where the hump peaks

A larger lambda shifts the hump toward longer maturities.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

taus = np.linspace(0.1, 30, 300)

hump_l1 = svensson_zero_rate(taus, 0, 0, 0.02, 0, 1.0, 5.0)
hump_l2 = svensson_zero_rate(taus, 0, 0, 0.02, 0, 3.0, 5.0)
hump_l3 = svensson_zero_rate(taus, 0, 0, 0.02, 0, 7.0, 5.0)

plt.figure(figsize=(9, 4))
plt.plot(taus, hump_l1 * 100, label='λ₁ = 1')
plt.plot(taus, hump_l2 * 100, label='λ₁ = 3')
plt.plot(taus, hump_l3 * 100, label='λ₁ = 7')
plt.xlabel('Maturity (years)')
plt.ylabel('Yield contribution (%)')
plt.title('Hump term β₂·(loading − exp) for different λ₁')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Realistic curve shapes

Three typical Treasury shapes — normal (upward sloping), inverted, and humped.

In [ ]:
taus = np.linspace(0.1, 30, 300)
maturities_label = [0.25, 1, 2, 5, 10, 30]

curves = {
    'Normal (upward sloping)':  (0.045, -0.020,  0.010, -0.005, 1.5, 4.0),
    'Inverted':                  (0.035,  0.020, -0.015,  0.005, 1.5, 4.0),
    'Humped':                    (0.040, -0.005,  0.030, -0.010, 1.0, 6.0),
}

plt.figure(figsize=(10, 5))
for label, params in curves.items():
    rates = svensson_zero_rate(taus, *params)
    plt.plot(taus, rates * 100, label=label)

plt.xlabel('Maturity (years)')
plt.ylabel('Zero-coupon yield (%)')
plt.title('Svensson curve — three typical shapes')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xticks(maturities_label)
plt.tight_layout()
plt.show()

## 6. Compare against Fed's published parameters

The `feds200628` dataset includes the fitted Svensson parameters for each date.
Pick one date and confirm our function reproduces their zero rates at standard maturities.

In [ ]:
import pandas as pd

fed_path = '../data/processed/treasury_bonds.parquet'
try:
    df = pd.read_parquet(fed_path)
    df['date'] = pd.to_datetime(df['date'])

    row = df.dropna(subset=['beta0','beta1','beta2','beta3','tau1','tau2']).iloc[-1]
    date = row['date']
    params = row['beta0'], row['beta1'], row['beta2'], row['beta3'], row['tau1'], row['tau2']
    print(f"Date: {date.date()}")
    print(f"b0={params[0]:.4f}, b1={params[1]:.4f}, b2={params[2]:.4f}, b3={params[3]:.4f}")
    print(f"l1={params[4]:.4f}, l2={params[5]:.4f}")

    # Fed stores params in percent units; svensson_zero_rate is pure arithmetic,
    # so output is also in percent — no need to multiply by 100.
    taus = np.array([1, 2, 3, 5, 7, 10, 20, 30])
    our_rates = svensson_zero_rate(taus, *params)
    fed_rates  = np.array([row[f"sveny{int(t):02d}"] for t in taus])

    print(f"
{'Maturity':<10} {'Our rate':<12} {'Fed rate':<12} {'Diff (bp)'}")
    for t, our, fed in zip(taus, our_rates, fed_rates):
        diff_bp = (our - fed) * 100
        print(f"  {t:>3.0f}y      {our:>7.4f}%    {fed:>7.4f}%    {diff_bp:+.2f}")
except FileNotFoundError:
    print("treasury_bonds.parquet not found — run load_fed_curves() first")
